# Transformar Datos Attendances 
- Extraer la fecha  de campo check_in en un nuevo campo llamado date
- Extaer la hora del campo check_in y  check_out y llamar check_in_time y check_out_time
- Asignar al campus status los siguientes 1:Present 2:Absent 3:Remote 4:Late 5:Vacation 6:On Leave
- Escribir los datos en la capa silver

In [ ]:
df_attendances=spark.table("zenviro.bronze.attendances_py")

In [ ]:
from pyspark.sql.functions import date_format,col,when

In [ ]:
df_attendances_1=df_attendances.select(
   "attendances_id",
    "employees_id",
    date_format("check_in","yyyy-MM-dd").cast("date").alias("date"),
    "check_in",
    "check_out",
    "status"
)

In [ ]:
df_attendances_1=df_attendances.select(
   "attendances_id",
    "employees_id",
    date_format("check_in","yyyy-MM-dd").cast("date").alias("date"),
    date_format("check_in","HH:mm:ss").cast("timestamp").alias("check_in_time"),
    date_format("check_out","HH:mm:ss").cast("timestamp").alias("check_out_time"),
    "status"
)

In [ ]:
df_attendances_status=(
    df_attendances_1
    .withColumn("status",when(col("status")=="1","Present")
    .when(col("status")=="2","Absent")
    .when(col("status")=="3","Remote")
    .when(col("status")=="4","Vacation")
    .when(col("status")=="5","On Leave")
    .otherwise("Unknown")).alias("status")
)

In [ ]:
df_attendances_status.writeTo("zenviro.silver.attendances_py").createOrReplace()